Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load the Merged Dataset

In [3]:
merged_df = pd.read_csv(
    "../datasets/cleaned/merged_customer_data.csv",
    parse_dates=["order_purchase_timestamp"]
)

merged_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_item_id,product_id,price,freight_value,payment_type,payment_installments,payment_value
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-06-05,1,a9516a079e37a9c9c36b9b78b10169e8,124.99,21.88,credit_card,2,146.87
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-02-06,1,4aa6014eceb682077f9dc4bffebc05b0,289.00,46.48,credit_card,8,335.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-06-13,1,bd07b66896d6f1494f5b86251848ced7,139.94,17.79,credit_card,7,157.73
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-04-10,1,a5647c44af977b148e0a3a4751a09e2e,149.94,23.36,credit_card,1,173.30
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-08-15,1,9391a573abe00141c56e38d84d7d5b3b,230.00,22.25,credit_card,8,252.25


Choose a Reference Date

In [4]:
reference_date = merged_df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

print(reference_date)

2018-08-30 15:00:37


Create RFM Features

Recency

In [5]:
recency = (
    merged_df.groupby("customer_unique_id")["order_purchase_timestamp"]
    .max()
    .reset_index()
)

recency["Recency"] = (
    reference_date - recency["order_purchase_timestamp"]
).dt.days

recency = recency.drop(columns="order_purchase_timestamp")

recency.head()

,customer_unique_id,Recency
0,0000366f3b9a7992bf8c76cfdf3221e2,112
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115
2,0000f46a3911fa3c0805444483337064,537
3,0000f6ccb0745a6a4b88665a16c9f078,321
4,0004aac84e0df4da2b147fca70cf8255,288


Frequency

In [6]:
frequency = (
    merged_df.groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index(name="Frequency")
)

frequency.head()

,customer_unique_id,Frequency
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


Monetary

In [7]:
monetary = (
    merged_df.groupby("customer_unique_id")["payment_value"]
    .sum()
    .reset_index(name="Monetary")
)

monetary.head()

,customer_unique_id,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


Additional Customer Features

Total Freight

In [8]:
freight = (
    merged_df.groupby("customer_unique_id")["freight_value"]
    .sum()
    .reset_index(name="Total_Freight")
)

Average Product Price

In [9]:
avg_price = (
    merged_df.groupby("customer_unique_id")["price"]
    .mean()
    .reset_index(name="Average_Product_Price")
)

Total Products Purchased

In [10]:
products = (
    merged_df.groupby("customer_unique_id")["order_item_id"]
    .count()
    .reset_index(name="Total_Products")
)

Average Payment Installments

In [11]:
installments = (
    merged_df.groupby("customer_unique_id")["payment_installments"]
    .mean()
    .reset_index(name="Average_Installments")
)

Preferred Payment Method

In [12]:
payment_type = (
    merged_df.groupby("customer_unique_id")["payment_type"]
    .agg(lambda x: x.mode()[0])
    .reset_index(name="Preferred_Payment")
)

Merge All Features

In [13]:
customer_features = recency.merge(
    frequency,
    on="customer_unique_id"
)

customer_features = customer_features.merge(
    monetary,
    on="customer_unique_id"
)

customer_features = customer_features.merge(
    freight,
    on="customer_unique_id"
)

customer_features = customer_features.merge(
    avg_price,
    on="customer_unique_id"
)

customer_features = customer_features.merge(
    products,
    on="customer_unique_id"
)

customer_features = customer_features.merge(
    installments,
    on="customer_unique_id"
)

customer_features = customer_features.merge(
    payment_type,
    on="customer_unique_id"
)

customer_features.head()

,customer_unique_id,Recency,Frequency,Monetary,Total_Freight,Average_Product_Price,Total_Products,Average_Installments,Preferred_Payment
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,12.00,129.90,1,8.0,credit_card
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,8.29,18.90,1,1.0,credit_card
2,0000f46a3911fa3c0805444483337064,537,1,86.22,17.22,69.00,1,8.0,credit_card
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,17.63,25.99,1,4.0,credit_card
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,16.89,180.00,1,6.0,credit_card


Check the Dataset

In [14]:
customer_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93335 entries, 0 to 93334
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_unique_id     93335 non-null  object 
 1   Recency                93335 non-null  int64  
 2   Frequency              93335 non-null  int64  
 3   Monetary               93335 non-null  float64
 4   Total_Freight          93335 non-null  float64
 5   Average_Product_Price  93335 non-null  float64
 6   Total_Products         93335 non-null  int64  
 7   Average_Installments   93335 non-null  float64
 8   Preferred_Payment      93335 non-null  object 
dtypes: float64(4), int64(3), object(2)
memory usage: 6.4+ MB


In [15]:
customer_features.describe()

,Recency,Frequency,Monetary,Total_Freight,Average_Product_Price,Total_Products,Average_Installments
count,93335.000000,93335.000000,93335.000000,93335.000000,93335.000000,93335.000000,93335.000000
mean,237.898752,1.033417,211.844495,24.639861,125.831300,1.232239,2.900600
std,152.545822,0.209099,642.239857,27.057386,190.553959,0.819845,2.677201
min,1.000000,1.000000,9.590000,0.000000,0.850000,1.000000,0.000000
25%,114.000000,1.000000,63.750000,14.100000,42.900000,1.000000,1.000000
50%,219.000000,1.000000,112.950000,17.680000,79.000000,1.000000,2.000000
75%,346.000000,1.000000,201.740000,26.550000,139.900000,1.000000,4.000000
max,695.000000,15.000000,109312.640000,1794.960000,6735.000000,75.000000,24.000000


In [16]:
customer_features.isnull().sum()

customer_unique_id       0
Recency                  0
Frequency                0
Monetary                 0
Total_Freight            0
Average_Product_Price    0
Total_Products           0
Average_Installments     0
Preferred_Payment        0
dtype: int64

Save the Feature Dataset

In [17]:
customer_features.to_csv(
    "../datasets/cleaned/customer_features.csv",
    index=False
)

print("Customer feature dataset saved successfully!")

Customer feature dataset saved successfully!
